In [35]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import (ColumnDataSource, HoverTool, Span, Label, Band, BoxAnnotation, Arrow, NormalHead, LabelSet)
from bokeh.models import Range1d, NumeralTickFormatter
from bokeh.layouts import column, row

In [36]:
def data_quality_report(df):
    report = {
        "Rows and Columns": df.shape,
        "Duplicates": df.duplicated().sum(),
        "Missing values": df.isnull().sum()
    }

    return report

df = pd.read_csv("data/gsw_vs_league.csv")
report = data_quality_report(df)
print(report)

{'Rows and Columns': (47, 31), 'Duplicates': np.int64(0), 'Missing values': Season                  0
Team                    0
3P                      0
3PA                     0
3P%                     0
2P                      0
2PA                     0
2P%                     0
FGA                     0
PTS                     0
Teams_count             0
Lg_3P                   0
Lg_3PA                  0
Lg_3P%                  0
Lg_2P                   0
Lg_2PA                  0
Lg_2P%                  0
Lg_FGA                  0
Lg_PTS                  0
Diff_3P                 0
Diff_3PA                0
Diff_3P%                0
Diff_2P                 0
Diff_2PA                0
Diff_2P%                0
Diff_FGA                0
Diff_PTS                0
GSW_3P_scoring_share    0
Lg_3P_scoring_share     0
GSW_3PA_share           0
Lg_3PA_share            0
dtype: int64}


## Defining Key Events and Annotations

In [37]:
# Key events to annotate on charts
key_events = {
    1980: "3PT Line\nIntroduced",
    1995: "3PT Line\nShortened",
    1997: "3PT Line\nRestored",
    2010: "Curry\nDrafted",
    2014: "1st\nChampionship",
    2016: "73-9\nSeason",
    2016: "KD\nJoins",
    2019: "Last\nTitle Run",
    2020: "Klay & Steph\nInjured",
    2022: "4th\nChampionship",
}

# Era definitions (start_year, end_year, label)
eras = [
    (1979, 1994, "Early Adoption Era"),
    (1994, 1997, "Short Line Era"),
    (1997, 2015, "Mid Range Era"),
    (2015, 2024, "Warriors Dynasty (2015–2024)"),
    (2024, 2026, "Modern NBA"),
]

# Color scheme
GSW_COLOR    = "#FFC72C"   # Warriors = gold
LEAGUE_COLOR = "#45cf40"   # Gray = league
BG_COLOR     = "#0b1120"   # Dark bg
PANEL_COLOR  = "#111827"   # Slightly lighter panel
WHITE        = "#f0f4ff"
RED          = "#ef4444"
GREEN        = "#22c55e"
BLUE         = "#3b82f6"

print("Key events, eras and colors defined")
print(f"{len(key_events)} key events")
print(f"{len(eras)} eras")

Key events, eras and colors defined
9 key events
5 eras


In [38]:
# Reconstructing true 30-team league average
# Formula: ((29-team avg * 29) + GSW value) / 30
df = pd.read_csv("data/gsw_vs_league.csv")

df['Year'] = df['Season'].str[:4].astype(int)

df['Full_Lg_3P']  = ((df['Lg_3P']  * df['Teams_count']) + df['3P'])  / (df['Teams_count'] + 1)
df['Full_Lg_3PA'] = ((df['Lg_3PA'] * df['Teams_count']) + df['3PA']) / (df['Teams_count'] + 1)
df['Full_Lg_3P%'] = ((df['Lg_3P%'] * df['Teams_count']) + df['3P%']) / (df['Teams_count'] + 1)
df['Full_Lg_2PA'] = ((df['Lg_2PA'] * df['Teams_count']) + df['2PA']) / (df['Teams_count'] + 1)
df['Full_Lg_PTS'] = ((df['Lg_PTS'] * df['Teams_count']) + df['PTS']) / (df['Teams_count'] + 1)
df['Full_Lg_FGA'] = ((df['Lg_FGA'] * df['Teams_count']) + df['FGA']) / (df['Teams_count'] + 1)

# Share columns
df['Full_Lg_3PA_share'] = (df['Full_Lg_3PA'] / df['Full_Lg_FGA']).round(3)
df['Full_Lg_3P_scoring_share'] = ((df['Full_Lg_3P'] * 3) / df['Full_Lg_PTS']).round(3)

# Round everything
for col in ['Full_Lg_3P','Full_Lg_3PA','Full_Lg_3P%','Full_Lg_2PA','Full_Lg_PTS','Full_Lg_FGA']:
    df[col] = df[col].round(3)

# Sanity check
print("Data loaded successfully")
print(f"   Seasons: {df['Season'].iloc[0]} to {df['Season'].iloc[-1]}")
print(f"   Total seasons: {len(df)}")
print(f"\nFull league avg top 5:")
print(df[['Season','Full_Lg_3PA','Lg_3PA','3PA']].head(5))

Data loaded successfully
   Seasons: 1979-80 to 2025-26
   Total seasons: 47

Full league avg top 5:
    Season  Full_Lg_3PA  Lg_3PA  3PA
0  1979-80        2.769   2.829  1.5
1  1980-81        2.017   1.991  2.6
2  1981-82        2.287   2.209  4.0
3  1982-83        2.252   2.273  1.8
4  1983-84        2.374   2.355  2.8


## Part 1

## Chart A: League-wide 3PA Over Time.

In [39]:
output_notebook()

source = ColumnDataSource(df)
p1 = figure(
    title="The Three-Point Revolution: League-Wide 3PA Per Game (1979–2026)",
    width=950, height=500,
    x_range=(1979, 2027),
    tools="pan,wheel_zoom,box_zoom,reset,save",
    toolbar_location="above"
)

# style
p1.background_fill_color        = PANEL_COLOR
p1.border_fill_color            = BG_COLOR
p1.outline_line_color           = "#1f2937"
p1.title.text_color             = WHITE
p1.title.text_font              = "Georgia"
p1.title.text_font_size         = "14px"
p1.xaxis.major_label_text_color = LEAGUE_COLOR
p1.yaxis.major_label_text_color = LEAGUE_COLOR
p1.xaxis.axis_label             = "Season"
p1.yaxis.axis_label             = "3-Point Attempts Per Game"
p1.xaxis.axis_label_text_color  = LEAGUE_COLOR
p1.yaxis.axis_label_text_color  = LEAGUE_COLOR
p1.xgrid.grid_line_color        = "#1f2937"
p1.ygrid.grid_line_color        = "#1f2937"

# shading eras
era_colors = ["#0d1b2a","#1a1a2e","#0d1b2a","#1a1a2e","#0d1b2a","#1a1a2e"]
for (start, end, label), color in zip(eras, era_colors):
    p1.add_layout(BoxAnnotation(
        left=start, right=end,
        fill_color=color, fill_alpha=0.6
    ))
    p1.add_layout(Label(
        x=start + (end - start) / 2, y=36,
        text=label, text_color="#4b5563",
        text_font_size="8px", text_align="center"
    ))

p1.line('Year', 'Full_Lg_3PA', source=source, color=LEAGUE_COLOR, line_width=2.5, legend_label="League Average (All 30 Teams)")
p1.scatter('Year', 'Full_Lg_3PA', source=source,
          color=LEAGUE_COLOR, size=5, alpha=0.8)

# Rule changes
rule_changes = {
    1994: ("3PT Line Shortened", RED),
    1997: ("3PT Line Restored",  GREEN),
}
for yr, (lbl, clr) in rule_changes.items():
    p1.add_layout(Span(
        location=yr, dimension='height',
        line_color=clr, line_dash='dashed', line_width=1.5
    ))
    p1.add_layout(Label(
        x=yr + 0.2, y=28,
        text=lbl, text_color=clr,
        text_font_size="13px", angle=1.5708
    ))

# Hover
p1.add_tools(HoverTool(tooltips=[
    ("Season",       "@Season"),
    ("League 3PA",   "@Full_Lg_3PA"),
    ("League 3P%",   "@{Full_Lg_3P%}{0.3%}"),
], mode='vline'))

# Legend
p1.legend.background_fill_color = PANEL_COLOR
p1.legend.label_text_color      = WHITE
p1.legend.border_line_color     = "#1f2937"
p1.legend.location              = "bottom_right"

show(p1)

Loading BokehJS ...

## Chart B: 3PA vs 2PA over time

In [27]:
df_3PA_vs_2PA = pd.read_csv("data/gsw_vs_league.csv")

In [28]:
print(df_3PA_vs_2PA.head())

    Season                   Team   3P  3PA    3P%    2P   2PA    2P%   FGA  \
0  1979-80  Golden State Warriors  0.3  1.5  0.223  42.7  87.8  0.486  89.2   
1  1980-81  Golden State Warriors  0.7  2.6  0.286  42.7  86.3  0.495  88.8   
2  1981-82  Golden State Warriors  1.1  4.0  0.280  43.4  85.7  0.506  89.6   
3  1982-83  Golden State Warriors  0.4  1.8  0.227  43.8  89.7  0.488  91.6   
4  1983-84  Golden State Warriors  0.7  2.8  0.243  42.2  89.1  0.474  91.9   

     PTS  ...  Diff_3P%  Diff_2P  Diff_2PA  Diff_2P%  Diff_FGA  Diff_PTS  \
0  103.6  ...    -0.042   -0.162    -0.076    -0.002    -1.519    -6.014   
1  109.8  ...     0.059    0.241    -0.109     0.004     0.391     1.782   
2  110.9  ...     0.029    0.700    -0.273     0.009     1.445     2.418   
3  108.6  ...    -0.004    0.845     2.395    -0.004     2.032     0.091   
4  109.9  ...     0.001   -0.709     3.259    -0.026     3.705    -0.227   

   GSW_3P_scoring_share  Lg_3P_scoring_share  GSW_3PA_share  Lg_3PA_

In [29]:
source = ColumnDataSource(df)

p2 = figure(title="The Declining 2PA: 3PA vs 2PA League-Wide Per Game (1979–2026)", width=950, height=420, x_range=(1979, 2027),
)

p2.background_fill_color        = PANEL_COLOR
p2.border_fill_color            = BG_COLOR
p2.outline_line_color           = "#1f2937"
p2.title.text_color             = WHITE
p2.title.text_font              = "Georgia"
p2.title.text_font_size         = "14px"
p2.xaxis.major_label_text_color = LEAGUE_COLOR
p2.yaxis.major_label_text_color = LEAGUE_COLOR
p2.xaxis.axis_label             = "Season"
p2.yaxis.axis_label             = "Field Goal Attempts Per Game"
p2.xaxis.axis_label_text_color  = LEAGUE_COLOR
p2.yaxis.axis_label_text_color  = LEAGUE_COLOR
p2.xgrid.grid_line_color        = "#1f2937"
p2.ygrid.grid_line_color        = "#1f2937"


# 2PA line — declining
p2.line('Year', 'Full_Lg_2PA', source=source,
        color=RED, line_width=2.5,
        legend_label="2-Point Attempts (2PA)")
p2.scatter('Year', 'Full_Lg_2PA', source=source,
          color=RED, size=5, alpha=0.8)

# 3PA line — rising
p2.line('Year', 'Full_Lg_3PA', source=source,
        color=GSW_COLOR, line_width=2.5,
        legend_label="3-Point Attempts (3PA)")
p2.scatter('Year', 'Full_Lg_3PA', source=source,
          color=GSW_COLOR, size=5, alpha=0.8)

# Find and annotate the crossover point
# Where 3PA first exceeds 2PA
crossover = df[df['Full_Lg_3PA'] >= df['Full_Lg_2PA']].head(1)
if not crossover.empty:
    cross_year = crossover['Year'].values[0]
    cross_val  = crossover['Full_Lg_3PA'].values[0]
    cross_span = Span(
        location=cross_year, dimension='height',
        line_color=WHITE, line_dash='dashed', line_width=1.5
    )
    p2.add_layout(cross_span)
    p2.add_layout(Label(
        x=cross_year + 0.3, y=cross_val + 2,
        text=f"Crossover\n{crossover['Season'].values[0]}",
        text_color=WHITE, text_font_size="9px"
    ))

# Rule change lines
for yr, (lbl, clr) in rule_changes.items():
    p2.add_layout(Span(
        location=yr, dimension='height',
        line_color=clr, line_dash='dashed', line_width=1.5
    ))

p2.add_tools(HoverTool(tooltips=[
    ("Season", "@Season"),
    ("3PA",    "@Full_Lg_3PA"),
    ("2PA",    "@Full_Lg_2PA"),
], mode='vline'))

p2.legend.background_fill_color = PANEL_COLOR
p2.legend.label_text_color      = WHITE
p2.legend.location              = "top_right"

show(p2)

In [30]:
source = ColumnDataSource(df)
p3 = figure(
    title="3-Pointers as a Share of Total Scoring: League-Wide (1979–2026)",
    width=950, height=400,
    x_range=(1979, 2027),
    tools="pan,wheel_zoom,box_zoom,reset,save",
    toolbar_location="above"
)

# Styling
p3.background_fill_color        = PANEL_COLOR
p3.border_fill_color            = BG_COLOR
p3.outline_line_color           = "#1f2937"
p3.title.text_color             = WHITE
p3.title.text_font              = "Georgia"
p3.title.text_font_size         = "14px"
p3.xaxis.major_label_text_color = LEAGUE_COLOR
p3.yaxis.major_label_text_color = LEAGUE_COLOR
p3.xaxis.axis_label             = "Season"
p3.yaxis.axis_label             = "Share of Total Points from 3s"
p3.xaxis.axis_label_text_color  = WHITE
p3.yaxis.axis_label_text_color  = WHITE
p3.xgrid.grid_line_color        = "#1f2937"
p3.ygrid.grid_line_color        = "#1f2937"
p3.yaxis.formatter              = NumeralTickFormatter(format="0%")


# Area fill under the line
band_source3 = ColumnDataSource(dict(
    Year  = df['Year'].tolist(),
    upper = df['Full_Lg_3P_scoring_share'].tolist(),
    lower = [0] * len(df)
))
p3.add_layout(Band(
    base='Year', upper='upper', lower='lower',
    source=band_source3,
    fill_color=GSW_COLOR, fill_alpha=0.15, line_alpha=0
))

# Main line
p3.line('Year', 'Full_Lg_3P_scoring_share', source=source,
        color=GSW_COLOR, line_width=2.5,
        legend_label="% of Points from 3s")
p3.scatter('Year', 'Full_Lg_3P_scoring_share', source=source,
           color=GSW_COLOR, size=5, alpha=0.8)

# Rule change lines
for yr, (lbl, clr) in rule_changes.items():
    p3.add_layout(Span(
        location=yr, dimension='height',
        line_color=clr, line_dash='dashed', line_width=1.5
    ))

# Annotate first and last values
first = df.iloc[0]
last  = df.iloc[-1]
p3.add_layout(Label(
    x=first['Year'] + 0.5,
    y=first['Full_Lg_3P_scoring_share'] + 0.01,
    text=f"{first['Full_Lg_3P_scoring_share']:.1%}",
    text_color=WHITE, text_font_size="9px"
))
p3.add_layout(Label(
    x=last['Year'] - 2,
    y=last['Full_Lg_3P_scoring_share'] + 0.01,
    text=f"{last['Full_Lg_3P_scoring_share']:.1%}",
    text_color=WHITE, text_font_size="9px"
))

# Hover
p3.add_tools(HoverTool(tooltips=[
    ("Season",          "@Season"),
    ("3P Scoring Share","@{Full_Lg_3P_scoring_share}{0.1%}"),
], mode='vline'))

# Legend
p3.legend.background_fill_color = PANEL_COLOR
p3.legend.label_text_color      = WHITE
p3.legend.border_line_color     = "#1f2937"
p3.legend.location              = "top_left"

show(p3)

## Act 2: The Warriors Story

In [32]:
source = ColumnDataSource(df)
p4 = figure(
    title="Golden State Warriors vs. Rest of League: 3-Pointers Made Per Game (1979–2026)",
    width=950, height=550,
    x_range=(1979, 2027),
    tools="pan,wheel_zoom,box_zoom,reset,save",
    toolbar_location="above"
)

# Styling
p4.background_fill_color        = PANEL_COLOR
p4.border_fill_color            = BG_COLOR
p4.outline_line_color           = "#1f2937"
p4.title.text_color             = WHITE
p4.title.text_font              = "Georgia"
p4.title.text_font_size         = "14px"
p4.xaxis.major_label_text_color = LEAGUE_COLOR
p4.yaxis.major_label_text_color = LEAGUE_COLOR
p4.xaxis.axis_label             = "Season"
p4.yaxis.axis_label             = "3-Pointers Made Per Game"
p4.xaxis.axis_label_text_color  = WHITE
p4.yaxis.axis_label_text_color  = WHITE
p4.xgrid.grid_line_color        = "#1f2937"
p4.ygrid.grid_line_color        = "#1f2937"

# Shaded gap between GSW and league
band_source4 = ColumnDataSource(dict(
    Year  = df['Year'].tolist(),
    upper = df[['3P', 'Lg_3P']].max(axis=1).tolist(),
    lower = df[['3P', 'Lg_3P']].min(axis=1).tolist(),
))
p4.add_layout(Band(
    base='Year', upper='upper', lower='lower',
    source=band_source4,
    fill_color=GSW_COLOR, fill_alpha=0.15, line_alpha=0
))

# League average line
p4.line('Year', 'Lg_3P', source=source,
        color=LEAGUE_COLOR, line_width=2,
        line_dash='dashed',
        legend_label="League Avg (excl. GSW)")
p4.scatter('Year', 'Lg_3P', source=source,
           color=LEAGUE_COLOR, size=4, alpha=0.7)

# GSW line — gold
p4.line('Year', '3P', source=source,
        color=GSW_COLOR, line_width=3,
        legend_label="Golden State Warriors")
p4.scatter('Year', '3P', source=source,
           color=GSW_COLOR, size=6, alpha=0.9)

# Dotted vertical lines for key events
for yr, lbl in key_events.items():
    p4.add_layout(Span(
        location=yr, dimension='height',
        line_color=WHITE, line_dash='dashed',
        line_width=1, line_alpha=0.25
    ))
    p4.add_layout(Label(
        x=yr + 0.2, y=0.8,
        text=lbl.replace('\n', ' '),
        text_color=WHITE, text_font_size="8px",
        angle=1.5708,
        background_fill_color=PANEL_COLOR,
        background_fill_alpha=0.6
    ))

# Arrow annotations on top 3 outlier seasons
top_outliers = df.nlargest(3, 'Diff_3P')
for _, r in top_outliers.iterrows():
    yr  = r['Year']
    val = r['3P']
    p4.add_layout(Arrow(
        end=NormalHead(fill_color=GSW_COLOR, size=8, line_color=GSW_COLOR),
        x_start=yr, y_start=val + 3.5,
        x_end=yr,   y_end=val + 0.4,
        line_color=GSW_COLOR, line_width=1.5
    ))
    p4.add_layout(Label(
        x=yr, y=val + 3.7,
        text=r['Season'],
        text_color=GSW_COLOR,
        text_font_size="9px",
        text_align="center"
    ))

p4.add_tools(HoverTool(tooltips=[
    ("Season",     "@Season"),
    ("GSW 3PM",    "@{3P}"),
    ("League 3PM", "@Lg_3P"),
    ("Gap",        "@Diff_3P"),
], mode='vline'))

p4.legend.background_fill_color = PANEL_COLOR
p4.legend.label_text_color      = WHITE
p4.legend.border_line_color     = "#1f2937"
p4.legend.location              = "top_left"

show(p4)


In [11]:
# Color each bar based on whether GSW is above or below league
bar_colors = [GSW_COLOR if d >= 0 else RED for d in df['Diff_3P']]
df['bar_color'] = bar_colors

# Recreate source to include bar_color
source = ColumnDataSource(df)

p5 = figure(
    title="The GSW Gap: Difference in 3PM vs. League Average Per Game (1979–2026)",
    width=950, height=380,
    x_range=(1979, 2027),
    tools="pan,wheel_zoom,box_zoom,reset,save",
    toolbar_location="above"
)

# Styling
p5.background_fill_color        = PANEL_COLOR
p5.border_fill_color            = BG_COLOR
p5.outline_line_color           = "#1f2937"
p5.title.text_color             = WHITE
p5.title.text_font              = "Georgia"
p5.title.text_font_size         = "14px"
p5.xaxis.major_label_text_color = LEAGUE_COLOR
p5.yaxis.major_label_text_color = LEAGUE_COLOR
p5.xaxis.axis_label             = "Season"
p5.yaxis.axis_label             = "GSW 3PM minus League Avg"
p5.xaxis.axis_label_text_color  = WHITE
p5.yaxis.axis_label_text_color  = WHITE
p5.xgrid.grid_line_color        = "#1f2937"
p5.ygrid.grid_line_color        = "#1f2937"

# Zero line
p5.add_layout(Span(
    location=0, dimension='width',
    line_color=WHITE, line_dash='dashed',
    line_width=1, line_alpha=0.4
))

# Bars
p5.vbar(
    x='Year', top='Diff_3P', source=source,
    width=0.7, color='bar_color', alpha=0.85
)

# Key event lines
for yr in key_events:
    p5.add_layout(Span(
        location=yr, dimension='height',
        line_color=WHITE, line_dash='dashed',
        line_width=1, line_alpha=0.2
    ))

# Annotate the biggest positive gap
best = df.loc[df['Diff_3P'].idxmax()]
p5.add_layout(Label(
    x=best['Year'], y=best['Diff_3P'] + 0.15,
    text=f"Peak gap\n{best['Season']} (+{best['Diff_3P']})",
    text_color=GSW_COLOR, text_font_size="9px",
    text_align="center"
))

# Annotate biggest negative gap
worst = df.loc[df['Diff_3P'].idxmin()]
p5.add_layout(Label(
    x=worst['Year'], y=worst['Diff_3P'] - 0.4,
    text=f"{worst['Season']} ({worst['Diff_3P']})",
    text_color=RED, text_font_size="9px",
    text_align="center"
))

# Hover
p5.add_tools(HoverTool(tooltips=[
    ("Season",   "@Season"),
    ("GSW 3PM",  "@{3P}"),
    ("Lg Avg",   "@Lg_3P"),
    ("Gap",      "@Diff_3P"),
]))

show(p5)

In [34]:
p6 = figure(
    title="3-Point Efficiency: GSW vs. League Average 3P% (1979–2026)",
    width=950, height=400,
    x_range=(1979, 2027)
)

# Styling
p6.background_fill_color        = PANEL_COLOR
p6.border_fill_color            = BG_COLOR
p6.outline_line_color           = "#1f2937"
p6.title.text_color             = WHITE
p6.title.text_font              = "Georgia"
p6.title.text_font_size         = "14px"
p6.xaxis.major_label_text_color = LEAGUE_COLOR
p6.yaxis.major_label_text_color = LEAGUE_COLOR
p6.xaxis.axis_label             = "Season"
p6.yaxis.axis_label             = "3-Point Percentage (3P%)"
p6.xaxis.axis_label_text_color  = WHITE
p6.yaxis.axis_label_text_color  = WHITE
p6.xgrid.grid_line_color        = "#1f2937"
p6.ygrid.grid_line_color        = "#1f2937"
p6.yaxis.formatter              = NumeralTickFormatter(format="0.0%")

# Era shading
for (start, end, label), color in zip(eras, era_colors):
    p6.add_layout(BoxAnnotation(
        left=start, right=end,
        fill_color=color, fill_alpha=0.6
    ))

# Shaded band between lines
band_source6 = ColumnDataSource(dict(
    Year  = df['Year'].tolist(),
    upper = df[['3P%', 'Lg_3P%']].max(axis=1).tolist(),
    lower = df[['3P%', 'Lg_3P%']].min(axis=1).tolist(),
))
p6.add_layout(Band(
    base='Year', upper='upper', lower='lower',
    source=band_source6,
    fill_color=GSW_COLOR, fill_alpha=0.12, line_alpha=0
))

# League average line
p6.line('Year', 'Lg_3P%', source=source,
        color=LEAGUE_COLOR, line_width=2,
        line_dash='dashed',
        legend_label="League Avg (excl. GSW)")
p6.scatter('Year', 'Lg_3P%', source=source,
           color=LEAGUE_COLOR, size=4, alpha=0.7)

# GSW line
p6.line('Year', '3P%', source=source,
        color=GSW_COLOR, line_width=3,
        legend_label="Golden State Warriors")
p6.scatter('Year', '3P%', source=source,
           color=GSW_COLOR, size=6, alpha=0.9)

# Key event lines
for yr in key_events:
    p6.add_layout(Span(
        location=yr, dimension='height',
        line_color=WHITE, line_dash='dashed',
        line_width=1, line_alpha=0.2
    ))

# Annotate GSW peak efficiency season
best_pct = df.loc[df['3P%'].idxmax()]
p6.add_layout(Label(
    x=best_pct['Year'], y=best_pct['3P%'] + 0.005,
    text=f"GSW peak\n{best_pct['Season']} ({best_pct['3P%']:.1%})",
    text_color=GSW_COLOR, text_font_size="9px",
    text_align="center"
))

# Hover
p6.add_tools(HoverTool(tooltips=[
    ("Season",     "@Season"),
    ("GSW 3P%",    "@{3P%}{0.3%}"),
    ("League 3P%", "@{Lg_3P%}{0.3%}"),
    ("Gap",        "@{Diff_3P%}{+0.3%}"),
], mode='vline'))

# Legend
p6.legend.background_fill_color = PANEL_COLOR
p6.legend.label_text_color      = WHITE
p6.legend.border_line_color     = "#1f2937"
p6.legend.location              = "top_left"

show(p6)

In [13]:
# Calculate rolling average to smooth the league trend
df['Lg_3PA_rolling'] = df['Full_Lg_3PA'].rolling(window=3, center=True).mean()

# Dynasty start year
DYNASTY_START = 2015

# Split data into pre and post dynasty
pre  = df[df['Year'] <  DYNASTY_START]
post = df[df['Year'] >= DYNASTY_START]

pre_source  = ColumnDataSource(pre)
post_source = ColumnDataSource(post)

p7 = figure(
    title="Did the Warriors Lead the Revolution? League 3PA Trend Before vs. After Dynasty (1979–2026)",
    width=950, height=500,
    x_range=(1979, 2027),
    tools="pan,wheel_zoom,box_zoom,reset,save",
    toolbar_location="above"
)

# Styling
p7.background_fill_color        = PANEL_COLOR
p7.border_fill_color            = BG_COLOR
p7.outline_line_color           = "#1f2937"
p7.title.text_color             = WHITE
p7.title.text_font              = "Georgia"
p7.title.text_font_size         = "14px"
p7.xaxis.major_label_text_color = LEAGUE_COLOR
p7.yaxis.major_label_text_color = LEAGUE_COLOR
p7.xaxis.axis_label             = "Season"
p7.yaxis.axis_label             = "League 3PA Per Game (Full League)"
p7.xaxis.axis_label_text_color  = WHITE
p7.yaxis.axis_label_text_color  = WHITE
p7.xgrid.grid_line_color        = "#1f2937"
p7.ygrid.grid_line_color        = "#1f2937"

# Shade pre dynasty era
p7.add_layout(BoxAnnotation(
    left=1979, right=DYNASTY_START,
    fill_color="#0d1b2a", fill_alpha=0.6
))
p7.add_layout(Label(
    x=1996, y=33,
    text="Pre-Dynasty Era",
    text_color="#4b5563", text_font_size="10px",
    text_align="center"
))

# Shade post dynasty era in gold tint
p7.add_layout(BoxAnnotation(
    left=DYNASTY_START, right=2027,
    fill_color="#2a1f00", fill_alpha=0.4
))
p7.add_layout(Label(
    x=2020, y=33,
    text="Post-Dynasty Era",
    text_color=GSW_COLOR, text_font_size="10px",
    text_align="center"
))

# Dynasty start line
p7.add_layout(Span(
    location=DYNASTY_START, dimension='height',
    line_color=GSW_COLOR, line_dash='solid',
    line_width=2, line_alpha=0.8
))
p7.add_layout(Label(
    x=DYNASTY_START + 0.2, y=20,
    text="Warriors\nDynasty Begins",
    text_color=GSW_COLOR, text_font_size="9px"
))

# Pre dynasty league trend — gray
p7.line('Year', 'Full_Lg_3PA', source=pre_source,
        color=LEAGUE_COLOR, line_width=2.5,
        legend_label="League 3PA — Pre Dynasty")
p7.scatter('Year', 'Full_Lg_3PA', source=pre_source,
           color=LEAGUE_COLOR, size=5, alpha=0.8)

# Post dynasty league trend — gold
p7.line('Year', 'Full_Lg_3PA', source=post_source,
        color=GSW_COLOR, line_width=2.5,
        legend_label="League 3PA — Post Dynasty")
p7.scatter('Year', 'Full_Lg_3PA', source=post_source,
           color=GSW_COLOR, size=5, alpha=0.8)

# GSW line for reference
p7.line('Year', '3PA', source=source,
        color=GSW_COLOR, line_width=1.5,
        line_dash='dashed', line_alpha=0.4,
        legend_label="GSW 3PA (reference)")

# Calculate and annotate rate of change
pre_slope  = (pre['Full_Lg_3PA'].iloc[-1]  - pre['Full_Lg_3PA'].iloc[0])  / len(pre)
post_slope = (post['Full_Lg_3PA'].iloc[-1] - post['Full_Lg_3PA'].iloc[0]) / len(post)

p7.add_layout(Label(
    x=1990, y=5,
    text=f"Pre-dynasty rate: +{pre_slope:.2f} 3PA/season",
    text_color=LEAGUE_COLOR, text_font_size="10px"
))
p7.add_layout(Label(
    x=2017, y=5,
    text=f"Post-dynasty rate: +{post_slope:.2f} 3PA/season",
    text_color=GSW_COLOR, text_font_size="10px"
))

# Hover
p7.add_tools(HoverTool(tooltips=[
    ("Season",      "@Season"),
    ("League 3PA",  "@Full_Lg_3PA"),
    ("GSW 3PA",     "@{3PA}"),
], mode='vline'))

# Legend
p7.legend.background_fill_color = PANEL_COLOR
p7.legend.label_text_color      = WHITE
p7.legend.border_line_color     = "#1f2937"
p7.legend.location              = "top_left"

show(p7)

In [40]:
df['Full_Lg_pts_from_3'] = (df['Full_Lg_3P'] * 3).round(2)
df['Full_Lg_pts_from_2'] = (df['Full_Lg_PTS'] - df['Full_Lg_pts_from_3']).round(2)

# Recreate source
source = ColumnDataSource(df)

# Quick check
p9 = figure(
    title="Where Are the Points Coming From? League-Wide Scoring Sources (1979–2026)",
    width=950, height=420,
    x_range=(1979, 2027),
    tools="pan,wheel_zoom,box_zoom,reset,save",
    toolbar_location="above"
)

# Styling
p9.background_fill_color        = PANEL_COLOR
p9.border_fill_color            = BG_COLOR
p9.outline_line_color           = "#1f2937"
p9.title.text_color             = WHITE
p9.title.text_font              = "Georgia"
p9.title.text_font_size         = "14px"
p9.xaxis.major_label_text_color = LEAGUE_COLOR
p9.yaxis.major_label_text_color = LEAGUE_COLOR
p9.xaxis.axis_label             = "Season"
p9.yaxis.axis_label             = "Points Per Game"
p9.xaxis.axis_label_text_color  = WHITE
p9.yaxis.axis_label_text_color  = WHITE
p9.xgrid.grid_line_color        = "#1f2937"
p9.ygrid.grid_line_color        = "#1f2937"

# Total points line — white
p9.line('Year', 'Full_Lg_PTS', source=source,
        color=WHITE, line_width=2,
        legend_label="Total Points Per Game")

# Points from twos — red declining
p9.line('Year', 'Full_Lg_pts_from_2', source=source,
        color=RED, line_width=2.5,
        legend_label="Points from Others")
p9.scatter('Year', 'Full_Lg_pts_from_2', source=source,
           color=RED, size=4, alpha=0.8)

# Points from threes — gold rising
p9.line('Year', 'Full_Lg_pts_from_3', source=source,
        color=GSW_COLOR, line_width=2.5,
        legend_label="Points from 3-Pointers")
p9.scatter('Year', 'Full_Lg_pts_from_3', source=source,
           color=GSW_COLOR, size=4, alpha=0.8)

# Annotate first and last values for 3s
first = df.iloc[0]
last  = df.iloc[-1]

p9.add_layout(Label(
    x=first['Year'] + 0.5,
    y=first['Full_Lg_pts_from_3'] + 1,
    text=f"{first['Full_Lg_pts_from_3']:.1f} pts",
    text_color=GSW_COLOR, text_font_size="9px"
))
p9.add_layout(Label(
    x=last['Year'] - 2,
    y=last['Full_Lg_pts_from_3'] + 1,
    text=f"{last['Full_Lg_pts_from_3']:.1f} pts",
    text_color=GSW_COLOR, text_font_size="9px"
))

# Annotate first and last values for 2s
p9.add_layout(Label(
    x=first['Year'] + 0.5,
    y=first['Full_Lg_pts_from_2'] + 1,
    text=f"{first['Full_Lg_pts_from_2']:.1f} pts",
    text_color=RED, text_font_size="9px"
))
p9.add_layout(Label(
    x=last['Year'] - 2,
    y=last['Full_Lg_pts_from_2'] + 1,
    text=f"{last['Full_Lg_pts_from_2']:.1f} pts",
    text_color=RED, text_font_size="9px"
))

# Rule change lines
for yr, (lbl, clr) in rule_changes.items():
    p9.add_layout(Span(
        location=yr, dimension='height',
        line_color=clr, line_dash='dashed', line_width=1.5
    ))

# Hover
p9.add_tools(HoverTool(tooltips=[
    ("Season",           "@Season"),
    ("Pts from 3s",      "@Full_Lg_pts_from_3"),
    ("Pts from 2s",      "@Full_Lg_pts_from_2"),
    ("Total Pts",        "@Full_Lg_PTS"),
], mode='vline'))

# Legend
p9.legend.background_fill_color = PANEL_COLOR
p9.legend.label_text_color      = WHITE
p9.legend.border_line_color     = "#1f2937"
p9.legend.location              = "bottom_right"

show(p9)